## 1. Install Dependencies

In [ ]:
# !pip install gymnasium[box2d]
# !pip install stable-baselines3[extra]
# !pip install moviepy

## 2. Imports

In [1]:
import gymnasium as gym
import numpy as np
import os

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import get_linear_fn

def linear_schedule(initial_value: float):
    """Linear schedule from initial_value down to 0."""
    return get_linear_fn(initial_value, 0.0, 1.0)

## 3. Create Environment




In [2]:
import gymnasium as gym
from gymnasium import Wrapper


class NegativeRewardTerminator(Wrapper):
    """
    Terminate the episode early if the agent accumulates too many
    consecutive negative-reward steps. This avoids wasting time on
    episodes where the car is hopelessly off-track.

    Parameters
    ----------
    env            : base environment
    patience       : how many consecutive negative-reward steps before termination
    negative_threshold : reward below this value counts as a "negative" step
    """
    def __init__(self, env, patience: int = 100, negative_threshold: float = 0.0):
        super().__init__(env)
        self.patience = patience
        self.negative_threshold = negative_threshold
        self._neg_count = 0

    def reset(self, **kwargs):
        self._neg_count = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        if reward < self.negative_threshold:
            self._neg_count += 1
        else:
            self._neg_count = 0

        if self._neg_count >= self.patience:
            terminated = True  
            self._neg_count = 0

        return obs, reward, terminated, truncated, info


def make_env(render_mode=None, use_terminator=True):
    def _init():
        env = gym.make("CarRacing-v3", continuous=True, render_mode=render_mode)
        if use_terminator:
            env = NegativeRewardTerminator(env, patience=100)
        env = Monitor(env)
        return env
    return _init


def build_env(n_envs=8, render_mode=None, n_stack=4, use_terminator=True):
    """Build a vectorized + frame-stacked environment."""
    env = DummyVecEnv([
        make_env(render_mode, use_terminator=(use_terminator and render_mode is None))
        for _ in range(n_envs)
    ])
    env = VecTransposeImage(env)
    env = VecFrameStack(env, n_stack=n_stack)
    return env


# Training environment — 8 parallel workers
N_ENVS = 8
train_env = build_env(n_envs=N_ENVS)

## 4. Create Model

**Key upgrades vs original:**

| Hyperparameter | Original | Optimized | Why |
|---|---|---|---|
| `n_steps` | 512 | 1024 | Larger rollouts → better advantage estimates |
| `batch_size` | 128 | 256 | Scales with `n_steps × n_envs` (8192 total); larger mini-batches are more stable |
| `n_epochs` | 4 | 10 | More passes over each rollout extracts more signal |
| `learning_rate` | linear 2.5e-4→0 | linear 3e-4→1e-5 | Doesn't decay to zero—preserves fine-tuning ability late in training |
| `ent_coef` | 0.01 (fixed) | 0.01 start, decayed via callback | SB3 PPO only accepts a float for ent_coef, so we decay it through a custom callback |
| `gae_lambda` | 0.95 | 0.95 | Kept — good default |
| `clip_range` | linear 0.2→0 | 0.2 (fixed) | Fixed clip range avoids over-restriction late in training |
| `vf_coef` | 0.5 | 0.5 | Kept |
| `max_grad_norm` | 0.5 | 0.5 | Kept |

In [ ]:

from stable_baselines3.common.callbacks import BaseCallback

class EntropyDecayCallback(BaseCallback):
    """Linearly decay ent_coef from start_val to end_val over total_timesteps."""
    def __init__(self, start_val=0.01, end_val=0.001, total_timesteps=5_000_000):
        super().__init__()
        self.start_val = start_val
        self.end_val = end_val
        self.total_timesteps = total_timesteps

    def _on_step(self) -> bool:
        progress = min(self.num_timesteps / self.total_timesteps, 1.0)
        self.model.ent_coef = self.start_val + progress * (self.end_val - self.start_val)
        return True


TOTAL_TIMESTEPS = 5_000_000

model = PPO(
    policy="CnnPolicy",
    env=train_env,

    learning_rate=get_linear_fn(3e-4, 1e-5, 1.0),

  
    n_steps=1024,
    batch_size=256,
    n_epochs=10,

   
    gamma=0.99,
    gae_lambda=0.95,

  
    clip_range=0.2,
    clip_range_vf=None,

    
    ent_coef=0.01,

    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    tensorboard_log="./ppo_car_tensorboard/",
    device="auto",
)

## 5. Train Model


- **5M timesteps** (up from 1M). With 8 envs this is ~625k environment "seconds" of wall-clock training, enough to reliably reach 900+.
- **`eval_freq=5_000`** (per env, so every ~40k global steps) — more frequent checkpointing of the best model.
- **`n_eval_episodes=10`** — more stable best-model selection.
- **`best_model_save_path`** unchanged — the `EvalCallback` still saves the best checkpoint automatically.

In [ ]:
os.makedirs("models/best", exist_ok=True)
os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("logs/eval", exist_ok=True)

eval_env_for_callback = build_env(n_envs=1, use_terminator=False)
eval_callback = EvalCallback(
    eval_env_for_callback,
    best_model_save_path="./models/best/",
    log_path="./logs/eval/",
    eval_freq=5_000,
    n_eval_episodes=10,
    deterministic=True,
    render=False,
)

checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path="./models/checkpoints/",
    name_prefix="ppo_car",
)


entropy_callback = EntropyDecayCallback(
    start_val=0.01,
    end_val=0.001,
    total_timesteps=TOTAL_TIMESTEPS,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_callback, checkpoint_callback, entropy_callback],
    reset_num_timesteps=True,
)
model.save("models/ppo_car_racing")

## 6. Continue Training from Checkpoint

Use this cell to resume training if you want to push past 5M steps.
Set `total_timesteps` to however many **additional** steps you want to run.

In [ ]:
# Rebuild env
train_env = build_env(n_envs=N_ENVS)

model = PPO.load(
    "models/ppo_car_racing",
    env=train_env,
    custom_objects={
        "learning_rate": get_linear_fn(3e-4, 1e-5, 1.0),
        "clip_range":    0.2,
        "ent_coef":      0.005,
    }
)

eval_env_for_callback = build_env(n_envs=1, use_terminator=False)
eval_callback = EvalCallback(
    eval_env_for_callback,
    best_model_save_path="./models/best/",
    log_path="./logs/eval/",
    eval_freq=5_000,
    n_eval_episodes=10,
    deterministic=True,
    render=False,
)


model.learn(
    total_timesteps=1_000_000,   
    callback=eval_callback,
    reset_num_timesteps=False,  
)

model.save("models/ppo_car_racing_v2")

## 7. Evaluate

Always load the **best** saved model (from `EvalCallback`) for final evaluation,
not the last checkpoint — the best checkpoint is often significantly higher.

In [3]:

eval_env = build_env(n_envs=1, use_terminator=False)  

best_model = PPO.load("models/best/best_model", env=eval_env)

episode_rewards, episode_lengths = evaluate_policy(
    best_model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True,
    return_episode_rewards=True,
)

eval_env.close()

mean_reward = np.mean(episode_rewards)
std_dev     = np.std(episode_rewards)
variance    = np.var(episode_rewards)
ci_95       = 1.96 * std_dev / np.sqrt(len(episode_rewards))

print(f"Episode Rewards : {[round(r, 2) for r in episode_rewards]}")
print(f"Mean Reward     : {mean_reward:.2f}")
print(f"Std Deviation   : {std_dev:.2f}")
print(f"Variance        : {variance:.2f}")
print(f"95% CI          : {mean_reward:.2f} ± {ci_95:.2f}")
print(f"Avg Ep Length   : {np.mean(episode_lengths):.1f} steps")

Episode Rewards : [938.8, 868.64, 112.69, 831.14, 748.8, 607.9, 865.4, 244.83, -67.59, 387.54]
Mean Reward     : 553.82
Std Deviation   : 341.48
Variance        : 116611.64
95% CI          : 553.82 ± 211.65
Avg Ep Length   : 879.2 steps


## 8. Visual Test (Human Render)


In [9]:
test_env = build_env(n_envs=1, render_mode="human", use_terminator=False)

obs = test_env.reset()

episodes = 5
ep = 0
ep_reward = 0.0

while ep < episodes:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    ep_reward += reward[0]

    if done[0]:
        ep += 1
        print(f"Episode {ep:2d} finished | Total Reward: {ep_reward:.2f}")
        ep_reward = 0.0
        obs = test_env.reset()

test_env.close()

Episode  1 finished | Total Reward: 901.90
Episode  2 finished | Total Reward: 935.10
Episode  3 finished | Total Reward: 890.26
Episode  4 finished | Total Reward: 871.52
Episode  5 finished | Total Reward: 243.54


## 9. Save Video Recording of Test Episodes

In [ ]:


import gymnasium as gym
import numpy as np
from gymnasium import Wrapper
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack, VecVideoRecorder
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import get_linear_fn


class NegativeRewardTerminator(Wrapper):
    def __init__(self, env, patience=100, negative_threshold=0.0):
        super().__init__(env)
        self.patience = patience
        self.negative_threshold = negative_threshold
        self._neg_count = 0

    def reset(self, **kwargs):
        self._neg_count = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        if reward < self.negative_threshold:
            self._neg_count += 1
        else:
            self._neg_count = 0
        if self._neg_count >= self.patience:
            terminated = True
            self._neg_count = 0
        return obs, reward, terminated, truncated, info


def make_env(render_mode=None, use_terminator=True):
    def _init():
        env = gym.make("CarRacing-v3", continuous=True, render_mode=render_mode)
        if use_terminator:
            env = NegativeRewardTerminator(env, patience=100)
        env = Monitor(env)
        return env
    return _init


def build_env(n_envs=1, render_mode=None, n_stack=4, use_terminator=True):
    env = DummyVecEnv([
        make_env(render_mode, use_terminator=(use_terminator and render_mode is None))
        for _ in range(n_envs)
    ])
    env = VecTransposeImage(env)
    env = VecFrameStack(env, n_stack=n_stack)
    return env


# Load best model
os.makedirs("./videos", exist_ok=True)
video_env = build_env(n_envs=1, render_mode="rgb_array", use_terminator=False)
best_model = PPO.load("models/best/best_model", env=video_env)

#  Wrap with video recorder
video_env = VecVideoRecorder(
    video_env,
    video_folder="./videos/",
    record_video_trigger=lambda step: step == 0,  
    video_length=3000,                             
    name_prefix="ppo_car_test"
)

# Record 
obs = video_env.reset()
for _ in range(3000):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, done, info = video_env.step(action)
    if done[0]:
        obs = video_env.reset()

video_env.close()
print("Video saved to ./videos/")

Saving video to d:\1.Projects\ReInforcmentLearning\AutonomousDriving\videos\ppo_car_test-step-0-to-step-3000.mp4
MoviePy - Building video d:\1.Projects\ReInforcmentLearning\AutonomousDriving\videos\ppo_car_test-step-0-to-step-3000.mp4.
MoviePy - Writing video d:\1.Projects\ReInforcmentLearning\AutonomousDriving\videos\ppo_car_test-step-0-to-step-3000.mp4



MoviePy - Done !
MoviePy - video ready d:\1.Projects\ReInforcmentLearning\AutonomousDriving\videos\ppo_car_test-step-0-to-step-3000.mp4
Video saved to ./videos/


## 10. Adding Obstacles (NPC Cars)

In [4]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from obstacle_wrapper import ObstacleWrapper, run_visual_test

def make_obstacle_env():
    def _init():
        env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")
        env = ObstacleWrapper(env, npc_penalty=-20.0, overtake_reward=50.0)
        env = Monitor(env)
        return env
    return _init

# Load model
best_model = PPO.load("models/best/best_model")

# Run visual test — pygame window will open
run_visual_test(best_model, make_obstacle_env, n_episodes=5)

Running 5 episodes

NPCs: 3 cars | 0.05 tiles/step | spawn: first at 20 tiles ahead, gap=15

  [NPC 0] spawned at step 0, tile 20/289 (217.8, 14.9)
  Delayed spawns: NPC 2 @ step 400 | NPC 3 @ step 700
  [NPC 0] ARMED — dist=4.7 tiles_ahead=288
  💥 [NPC 0] Collision! dist=0.81  -20
  [NPC 0] ARMED — tiles_ahead=1 dist=3.54 (need >=6 tiles AND dist>3.0)
  [NPC 0] ARMED — tiles_ahead=5 dist=17.99 (need >=6 tiles AND dist>3.0)
  ✅ [NPC 0] Overtake #1! +50  tiles_ahead=6  dist=19.4
  [step 200][NPC 0] dist=134.9 player_tile=69 npc_tile=30 tiles_ahead=39 state=AWARDED
  [step 200][NPC 1] dist=124.0 player_tile=69 npc_tile=0 tiles_ahead=69 state=WAITING
  [step 200][NPC 2] dist=124.0 player_tile=69 npc_tile=0 tiles_ahead=69 state=WAITING
  [step 400][NPC 0] dist=322.7 player_tile=159 npc_tile=39 tiles_ahead=120 state=AWARDED
  [step 400][NPC 1] dist=155.8 player_tile=159 npc_tile=0 tiles_ahead=159 state=WAITING
  [step 400][NPC 2] dist=155.8 player_tile=159 npc_tile=0 tiles_ahead=159 state=W